
# Raster Map Generation 0.1

*This notebook computes and visualizes the raster-based metrics from the original workflow. Configure the inputs, run the raster calculations, and then render the PNG maps with identical layouts and legends.*


## 1. Environment Setup

In [ ]:

# =============================================================================
# 1. Load required libraries
# =============================================================================
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import FuncFormatter, MaxNLocator

from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib_map_utils import north_arrow

from pyproj import CRS, Transformer
from tqdm.auto import tqdm

# Keep visualization consistent across all maps.
mpl.rcParams["font.family"] = "serif"


### 1.1 Provide file paths and analysis parameters

*Edit the next cell so it matches your folders, time points, and NoData value before running the rest of the notebook.*

In [ ]:

# =============================================================================
# 1. Define the analysis inputs (edit this cell)
# =============================================================================
# Provide the folders containing the raster time series for X and Y.
path_series_x = Path(
    r"C:\\Users\\AntFonseca\\github\\1.INPUT\\compare-time-series\\PIE\\pixelbased"
)
path_series_y = Path(
    r"C:\\Users\\AntFonseca\\github\\1.INPUT\\compare-time-series\\PIE\\objectbased"
)

# List the time points (years) to analyze.
time_points = [
    2010,
    2012,
    2014,
    2016,
    2018,
    2021
]

# Set a short class name used in filenames.
class_name = "PIE"

# Indicate whether the input rasters store binary (0/1) data.
is_binary_data = True

# Indicate the NoData value stored in the rasters.
nodata_value = 255

# Choose where the outputs will be stored.
output_path = Path(
    rf"C:\\Users\\AntFonseca\\github\\2.OUTPUT\\{class_name}"
)
output_path.mkdir(
    parents=True,
    exist_ok=True
)

print("? Parameters successfully defined.")


## 2. Helper Functions

In [ ]:

# =============================================================================
# 2. Helper functions (no need to edit)
# =============================================================================
MAP_SCALE_FACTOR = 0.15

raster_arrays = {}
_reference_profile_cache = None


def get_raster_array(year):
    # Return the X and Y rasters for a given year or None if missing.
    file_name = f"{class_name}{year}.tif"
    path_x = path_series_x / file_name
    path_y = path_series_y / file_name

    if year in raster_arrays:
        return raster_arrays[year]

    if not path_x.exists() or not path_y.exists():
        print(f"Warning: file '{file_name}' not found for year {year}.")
        return None, None

    with rasterio.open(path_x) as src_x:
        array_x = src_x.read(1)
    with rasterio.open(path_y) as src_y:
        array_y = src_y.read(1)

    raster_arrays[year] = (array_x, array_y)
    return array_x, array_y


def ensure_reference_profile():
    # Return the reference profile and raster dimensions.
    global _reference_profile_cache

    if _reference_profile_cache is not None:
        return _reference_profile_cache

    first_year = time_points[0]
    file_name = f"{class_name}{first_year}.tif"
    reference_path = path_series_x / file_name

    if not reference_path.exists():
        raise FileNotFoundError(
            f"Reference file not found: {reference_path}"
        )

    with rasterio.open(reference_path) as src:
        profile = src.profile.copy()
        height = src.height
        width = src.width

    _reference_profile_cache = (profile, height, width)
    return _reference_profile_cache


def decimal_to_dms(value):
    # Convert decimal degrees to degrees, minutes, seconds.
    degrees = int(abs(value))
    minutes_float = (abs(value) - degrees) * 60.0
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60.0
    return degrees, minutes, seconds


def make_lon_formatter(transformer, bounds):
    # Create a formatter for longitude tick labels.

    def _formatter(x_value, _):
        lon, _ = transformer.transform(
            x_value,
            bounds.bottom
        )
        degrees, minutes, seconds = decimal_to_dms(lon)
        suffix = "E" if lon >= 0 else "W"
        return (
            f"{degrees}° "
            f"{minutes}' "
            f"{seconds:.2f}\""
            f"{suffix}"
        )

    return FuncFormatter(_formatter)


def make_lat_formatter(transformer, bounds):
    # Create a formatter for latitude tick labels.

    def _formatter(y_value, _):
        _, lat = transformer.transform(
            bounds.left,
            y_value
        )
        degrees, minutes, seconds = decimal_to_dms(lat)
        suffix = "N" if lat >= 0 else "S"
        return (
            f"{degrees}° "
            f"{minutes}' "
            f"{seconds:.2f}\""
            f"{suffix}"
        )

    return FuncFormatter(_formatter)


def add_scalebar(ax, bounds, src_crs, transformer, length_fraction=0.4):
    # Add a consistent scale bar to the provided axes.
    crs_obj = CRS.from_user_input(src_crs)

    if crs_obj.is_geographic:
        center_x = (bounds.left + bounds.right) / 2.0
        center_y = (bounds.bottom + bounds.top) / 2.0
        lon_center, lat_center = transformer.transform(
            center_x,
            center_y
        )
        meters_per_degree_lon = 111_320.0 * np.cos(
            np.deg2rad(lat_center)
        )
        dx_km = meters_per_degree_lon / 1000.0
    else:
        try:
            meters_per_unit = crs_obj.axis_info[0].unit_conversion_factor
        except Exception:
            meters_per_unit = 1.0
        dx_km = meters_per_unit / 1000.0

    scalebar = ScaleBar(
        dx_km,
        units="km",
        length_fraction=length_fraction,
        location="lower right"
    )
    ax.add_artist(scalebar)


def prepare_map_axes(ax, bounds, src_crs, transformer):
    # Apply shared axis formatting, north arrow, and scalebar.
    ax.xaxis.set_major_formatter(
        make_lon_formatter(transformer, bounds)
    )
    ax.yaxis.set_major_formatter(
        make_lat_formatter(transformer, bounds)
    )
    ax.xaxis.set_major_locator(
        MaxNLocator(3)
    )
    ax.yaxis.set_major_locator(
        MaxNLocator(6)
    )
    ax.tick_params(
        axis="x",
        which="major",
        labelsize=10,
        pad=4
    )
    ax.tick_params(
        axis="y",
        which="major",
        labelsize=10,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )
    north_arrow(
        ax,
        location="upper right",
        rotation={"degrees": 0},
        shadow=False
    )
    add_scalebar(
        ax,
        bounds,
        src_crs,
        transformer
    )
    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)


def read_resampled_map(raster_path, scale_factor=MAP_SCALE_FACTOR):
    # Read a raster and resample it for plotting.
    with rasterio.open(raster_path) as src:
        bounds = src.bounds
        src_crs = src.crs
        transformer = Transformer.from_crs(
            src_crs,
            "EPSG:4326",
            always_xy=True
        )
        data = src.read(
            1,
            out_shape=(
                int(src.height * scale_factor),
                int(src.width * scale_factor)
            ),
            resampling=Resampling.nearest
        )

    masked = np.ma.masked_equal(
        data,
        nodata_value
    )
    return masked, bounds, src_crs, transformer


def blue_gray_red_cmap():
    # Return the diverging colormap used in change-difference plots.
    return mcolors.LinearSegmentedColormap.from_list(
        "red_gray_blue",
        [
            "#a50026",
            "#f46d43",
            "#f2f2f2",
            "#74add1",
            "#313695"
        ]
    )


## 3. Calculate Raster Maps

In [ ]:

# =============================================================================
# 3. Calculate raster maps
# =============================================================================
print("Starting raster generation...")

reference_profile, raster_height, raster_width = ensure_reference_profile()
profile_template = reference_profile.copy()


def write_raster(data_array, file_name):
    # Persist a float32 raster with LZW compression.
    profile = profile_template.copy()
    profile.update(
        dtype=rasterio.float32,
        nodata=nodata_value,
        compress="lzw",
        count=1
    )
    raster_path = output_path / file_name
    with rasterio.open(raster_path, "w", **profile) as dst:
        dst.write(
            data_array.astype(np.float32),
            1
        )
    return raster_path


def compute_presence_hit():
    # Compute the presence hit raster across all time points.
    print("\nCalculating Presence Hit raster...")
    accumulator = np.zeros(
        (raster_height, raster_width),
        dtype=np.float32
    )
    final_nodata_mask = np.ones(
        (raster_height, raster_width),
        dtype=bool
    )

    for year in tqdm(time_points, desc="Presence Hit", unit="year"):
        file_name = f"{class_name}{year}.tif"
        path_x = path_series_x / file_name
        path_y = path_series_y / file_name

        if not path_x.exists() or not path_y.exists():
            print(f"Warning: file '{file_name}' not found. Skipping.")
            continue

        with rasterio.open(path_x) as src_x, rasterio.open(path_y) as src_y:
            ax = src_x.read(1)
            ay = src_y.read(1)
            mask_x = src_x.read_masks(1) != 0
            mask_y = src_y.read_masks(1) != 0

            valid_mask = (
                mask_x
                & mask_y
                & ~(np.isnan(ax) | np.isnan(ay))
                & (ax != nodata_value)
                & (ay != nodata_value)
            )

            if is_binary_data:
                x_present = ax == 1
                y_present = ay == 1
                hit_map = (x_present & y_present).astype(np.float32)
            else:
                ax_pos = np.maximum(ax.astype(np.float32), 0.0)
                ay_pos = np.maximum(ay.astype(np.float32), 0.0)
                hit_map = np.minimum(ax_pos, ay_pos)

            np.add(
                accumulator,
                hit_map,
                out=accumulator,
                where=valid_mask
            )
            final_nodata_mask &= ~valid_mask

    accumulator[final_nodata_mask] = nodata_value
    raster_path = write_raster(
        accumulator,
        f"presence_hit_{class_name}.tif"
    )
    print(f"Saved: {raster_path}")
    return raster_path


def compute_presence_time_difference():
    # Compute Eq. 51 presence time difference raster.
    print("\nCalculating Presence Time Difference raster (Eq. 51)...")
    cn_map = np.zeros(
        (raster_height, raster_width),
        dtype=np.float32
    )
    prev_nonzero_sign = np.zeros(
        (raster_height, raster_width),
        dtype=np.int8
    )
    final_nodata_mask = np.ones(
        (raster_height, raster_width),
        dtype=bool
    )

    for year in tqdm(time_points, desc="Presence Time Difference", unit="year"):
        file_name = f"{class_name}{year}.tif"
        path_x = path_series_x / file_name
        path_y = path_series_y / file_name

        if not path_x.exists() or not path_y.exists():
            print(f"Warning: file '{file_name}' not found. Skipping.")
            continue

        with rasterio.open(path_x) as src_x, rasterio.open(path_y) as src_y:
            ax = src_x.read(1)
            ay = src_y.read(1)
            mask_x = src_x.read_masks(1) != 0
            mask_y = src_y.read_masks(1) != 0

            valid_mask = (
                mask_x
                & mask_y
                & ~(np.isnan(ax) | np.isnan(ay))
                & (ax != nodata_value)
                & (ay != nodata_value)
            )

            if is_binary_data:
                x_present = (ax == 1).astype(np.int8)
                y_present = (ay == 1).astype(np.int8)
                difference = y_present - x_present
            else:
                ax_pos = np.maximum(ax.astype(np.float32), 0.0)
                ay_pos = np.maximum(ay.astype(np.float32), 0.0)
                difference = ay_pos - ax_pos

            sign_curr = np.zeros_like(difference, dtype=np.int8)
            sign_curr[(difference > 0) & valid_mask] = 1
            sign_curr[(difference < 0) & valid_mask] = -1

            changed_mask = (
                (prev_nonzero_sign != 0)
                & (sign_curr != 0)
                & (sign_curr != prev_nonzero_sign)
                & valid_mask
            )

            np.add(
                cn_map,
                1.0,
                out=cn_map,
                where=changed_mask
            )

            prev_nonzero_sign = np.where(
                (sign_curr != 0) & valid_mask,
                sign_curr,
                prev_nonzero_sign
            ).astype(np.int8)

            final_nodata_mask &= ~valid_mask

    cn_map[final_nodata_mask] = nodata_value
    raster_path = write_raster(
        cn_map,
        f"presence_time_difference_{class_name}.tif"
    )
    print(f"Saved: {raster_path}")
    return raster_path


## 4. Visualize Raster Maps

In [ ]:

# =============================================================================
# 4. Visualize raster maps
# =============================================================================
print("Preparing map visualizations...")

def plot_presence_hit_map(raster_path):
    # Plot and export the Presence Hit map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    if is_binary_data:
        num_points = len(time_points)
        boundaries = np.arange(-0.5, num_points + 1.5, 1.0)

        viridis = mpl.colormaps["viridis"].resampled(
            max(num_points, 1)
        )
        colors_step = viridis(
            np.linspace(
                0.0,
                1.0,
                num_points,
                endpoint=True
            )
        )
        colors = ["#f2f2f2"] + [
            mcolors.to_hex(color)
            for color in colors_step
        ]
        cmap = ListedColormap(colors)
        cmap.set_bad(color="white")
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        colors = [
            "#f2f2f2",
            "#23b0f1",
            "#026092",
            "#051927"
        ]
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "gray_to_blue_hits",
            colors
        )
        cmap.set_bad(color="white")
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        norm = mcolors.Normalize(
            vmin=0.0,
            vmax=max_val
        )

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data:
        legend_labels = [
            str(value)
            for value in range(len(time_points) + 1)
        ]
        patches = [
            mpatches.Patch(
                color=colors[index],
                label=legend_labels[index]
            )
            for index in range(len(legend_labels))
        ]
        legend = ax.legend(
            handles=patches,
            title="Sum of Hits",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    else:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.5
        )
        colorbar.set_label(
            "Sum of Hits",
            fontsize=12,
            rotation=0,
            y=1.08,
            labelpad=0
        )
        colorbar.set_ticks(
            np.linspace(
                0.0,
                max_val,
                num=6
            )
        )

    ax.set_title(
        f"Presence Hits - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"presence_hit_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_presence_time_difference_map(raster_path):
    # Plot and export the Presence Time Difference map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    num_points = len(time_points)
    num_intervals = max(0, num_points - 1)
    boundaries = np.arange(-0.5, num_intervals + 1.5, 1.0)

    if num_intervals > 0:
        magma = mpl.colormaps["magma"].resampled(num_intervals)
        colors_step = magma(
            np.linspace(
                0.0,
                1.0,
                num_intervals,
                endpoint=True
            )
        )
        colors = ["#f2f2f2"] + [
            mcolors.to_hex(color)
            for color in colors_step
        ]
    else:
        colors = ["#f2f2f2"]

    cmap = ListedColormap(colors)
    cmap.set_bad(color="white")
    norm = BoundaryNorm(boundaries, cmap.N)

    ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    legend_labels = [
        str(value)
        for value in range(num_intervals + 1)
    ]
    patches = [
        mpatches.Patch(
            color=colors[index],
            label=legend_labels[index]
        )
        for index in range(len(legend_labels))
    ]
    legend = ax.legend(
        handles=patches,
        title="Time Difference",
        loc="center left",
        bbox_to_anchor=(1.05, 0.5),
        frameon=False,
        fontsize=12,
        alignment="left"
    )
    legend.get_title().set_fontsize("14")
    legend.get_title().set_ha("left")
    for item in legend.get_texts():
        item.set_ha("left")

    ax.set_title(
        f"Presence Time Difference - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"presence_temporal_disagreement_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_presence_difference_map(raster_path):
    # Plot and export the Presence Difference map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    if is_binary_data:
        if masked_map.count() > 0:
            min_val = int(np.floor(np.ma.min(masked_map)))
            max_val = int(np.ceil(np.ma.max(masked_map)))
        else:
            min_val, max_val = -1, 1

        if min_val < 0 and max_val > 0:
            sym = max(abs(min_val), abs(max_val))
            min_val = -sym
            max_val = sym

        colors_for_legend = []
        labels_for_legend = []

        if min_val < 0:
            neg_steps = abs(min_val)
            red_cmap = plt.get_cmap("Reds_r", max(neg_steps, 1))
            for index in range(neg_steps):
                fraction = 0.0 if neg_steps == 1 else index / (neg_steps - 1)
                colors_for_legend.append(
                    mcolors.to_hex(red_cmap(fraction))
                )
            labels_for_legend.extend(range(min_val, 0))

        if min_val <= 0 <= max_val:
            colors_for_legend.append("#f2f2f2")
            labels_for_legend.append(0)

        if max_val > 0:
            pos_steps = max_val
            blue_cmap = plt.get_cmap("Blues", max(pos_steps, 1))
            for index in range(pos_steps):
                fraction = 0.0 if pos_steps == 1 else index / (pos_steps - 1)
                colors_for_legend.append(
                    mcolors.to_hex(blue_cmap(fraction))
                )
            labels_for_legend.extend(range(1, max_val + 1))

        cmap = ListedColormap(colors_for_legend)
        cmap.set_bad(color="white")
        boundaries = np.arange(min_val - 0.5, max_val + 1.5, 1.0)
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        vmin = float(np.ma.min(masked_map)) if masked_map.count() > 0 else -1.0
        vmax = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        amax = max(abs(vmin), abs(vmax))
        norm = mcolors.Normalize(
            vmin=-amax,
            vmax=amax
        )
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "diff_diverging",
            [
                "#f4a582",
                "#d6604d",
                "#b2182b",
                "#f2f2f2",
                "#2166ac",
                "#4393c3",
                "#92c5de"
            ]
        )
        cmap.set_bad(color="white")

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data:
        patches = [
            mpatches.Patch(color=color, label=str(label))
            for color, label in zip(colors_for_legend, labels_for_legend)
        ]
        patches.reverse()
        legend = ax.legend(
            handles=patches,
            title="Difference",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    else:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.7
        )
        colorbar.set_label(
            "Presence Difference",
            fontsize=12,
            rotation=270,
            labelpad=20
        )

    ax.set_title(
        f"Presence Difference - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"presence_difference_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_presence_absolute_difference_map(raster_path):
    # Plot and export the Presence Absolute Difference map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    legend_colors = []
    legend_labels = []

    if is_binary_data:
        unique_vals = (
            sorted(np.unique(masked_map.compressed()).astype(int))
            if masked_map.count() > 0 else [0]
        )
        blue_cmap = plt.get_cmap("Blues")

        if 0 in unique_vals:
            legend_colors.append("#f2f2f2")
            legend_labels.append("0")

        positive_vals = [value for value in unique_vals if value > 0]
        count_pos = len(positive_vals)

        if count_pos > 0:
            blue_shades = [
                blue_cmap(fraction)
                for fraction in np.linspace(0.35, 1.0, count_pos)
            ]
            legend_colors.extend(
                [mcolors.to_hex(color) for color in blue_shades]
            )
            legend_labels.extend([str(value) for value in positive_vals])

        cmap = ListedColormap(
            legend_colors if legend_colors else ["#f2f2f2"]
        )
        cmap.set_bad(color="white")

        if unique_vals:
            boundaries = [value - 0.5 for value in unique_vals] + [unique_vals[-1] + 0.5]
        else:
            boundaries = [-0.5, 0.5]
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "gray_to_blue_abs",
            [
                "#f2f2f2",
                plt.get_cmap("Blues")(1.0)
            ]
        )
        cmap.set_bad(color="white")
        norm = mcolors.Normalize(
            vmin=0.0,
            vmax=max_val
        )

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data and legend_labels:
        patches = [
            mpatches.Patch(
                color=legend_colors[index],
                label=legend_labels[index]
            )
            for index in range(len(legend_labels))
        ]
        legend = ax.legend(
            handles=patches,
            title="Absolute Difference",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    elif not is_binary_data:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.7
        )
        colorbar.set_label(
            "Absolute Difference",
            fontsize=12,
            rotation=270,
            labelpad=20
        )
        colorbar.set_ticks(
            np.linspace(0.0, max_val, num=6)
        )

    ax.set_title(
        f"Presence Absolute Difference - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"presence_absolute_difference_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_change_hit_map(raster_path):
    # Plot and export the Change Hit map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    if is_binary_data:
        num_intervals = max(0, len(time_points) - 1)
        boundaries = np.arange(-0.5, num_intervals + 1.5, 1.0)

        base = mpl.colormaps["RdYlBu"]
        if num_intervals > 0:
            samples = np.linspace(0.0, 1.0, num_intervals, endpoint=True)
            interval_colors = [
                mcolors.to_hex(base(sample))
                for sample in samples
            ]
        else:
            interval_colors = []
        colors = ["#f2f2f2"] + interval_colors

        cmap = ListedColormap(colors)
        cmap.set_bad(color="white")
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        base = mpl.colormaps["RdYlBu"]
        colors = ["#f2f2f2"] + [
            mcolors.to_hex(base(sample))
            for sample in np.linspace(0.0, 1.0, 256)
        ]
        cmap = ListedColormap(colors)
        cmap.set_bad(color="white")
        norm = mcolors.Normalize(
            vmin=0.0,
            vmax=max_val
        )

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data:
        legend_labels = [
            str(value)
            for value in range(num_intervals + 1)
        ]
        patches = [
            mpatches.Patch(
                color=colors[index],
                label=legend_labels[index]
            )
            for index in range(len(legend_labels))
        ]
        legend = ax.legend(
            handles=patches,
            title="Change Hits",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    else:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.5
        )
        colorbar.set_label(
            "Change Hits",
            fontsize=12,
            rotation=270,
            labelpad=20
        )
        colorbar.set_ticks(
            np.linspace(
                0.0,
                max_val,
                num=6
            )
        )

    ax.set_title(
        f"Change Hits - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"change_hit_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path
def plot_change_difference_map(raster_path):
    # Plot and export the Change Difference map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    base_cmap = blue_gray_red_cmap()

    if is_binary_data:
        if masked_map.count() > 0:
            vmin = int(np.floor(np.ma.min(masked_map)))
            vmax = int(np.ceil(np.ma.max(masked_map)))
        else:
            vmin, vmax = -1, 1
        sym = max(abs(vmin), abs(vmax))
        values = list(range(-sym, sym + 1))
        colors = [
            "#f2f2f2" if value == 0 else mcolors.to_hex(base_cmap((value + sym) / (2 * sym)))
            for value in values
        ]
        cmap = ListedColormap(colors)
        cmap.set_bad(color="white")
        boundaries = np.arange(-sym - 0.5, sym + 1.5, 1.0)
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        vmin = float(np.ma.min(masked_map)) if masked_map.count() > 0 else -1.0
        vmax = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        amax = max(abs(vmin), abs(vmax))
        norm = mcolors.Normalize(
            vmin=-amax,
            vmax=amax
        )
        cmap = base_cmap
        cmap.set_bad(color="white")

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data:
        values_desc = list(range(sym, -sym - 1, -1))
        patches = [
            mpatches.Patch(
                color=mcolors.to_hex(cmap(norm(value))),
                label=str(value)
            )
            for value in values_desc
        ]
        legend = ax.legend(
            handles=patches,
            title="Change Difference",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    else:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.7
        )
        colorbar.set_label(
            "Change Difference",
            fontsize=12,
            rotation=270,
            labelpad=20
        )
        colorbar.set_ticks(
            np.linspace(-amax, amax, num=7)
        )

    ax.set_title(
        f"Change Difference - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"change_difference_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_change_absolute_difference_map(raster_path):
    # Plot and export the Change Absolute Difference map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    legend_colors = []
    legend_labels = []

    if is_binary_data:
        unique_vals = (
            sorted(np.unique(masked_map.compressed()).astype(int))
            if masked_map.count() > 0 else [0]
        )
        blue_cmap = plt.get_cmap("Blues")

        if 0 in unique_vals:
            legend_colors.append("#f2f2f2")
            legend_labels.append("0")

        positive_vals = [value for value in unique_vals if value > 0]
        count_pos = len(positive_vals)

        if count_pos > 0:
            blue_shades = [
                blue_cmap(fraction)
                for fraction in np.linspace(0.35, 1.0, count_pos)
            ]
            legend_colors.extend(
                [mcolors.to_hex(color) for color in blue_shades]
            )
            legend_labels.extend([str(value) for value in positive_vals])

        cmap = ListedColormap(
            legend_colors if legend_colors else ["#f2f2f2"]
        )
        cmap.set_bad(color="white")

        if unique_vals:
            boundaries = [value - 0.5 for value in unique_vals] + [unique_vals[-1] + 0.5]
        else:
            boundaries = [-0.5, 0.5]
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "gray_to_blue_abs",
            [
                "#f2f2f2",
                plt.get_cmap("Blues")(1.0)
            ]
        )
        cmap.set_bad(color="white")
        norm = mcolors.Normalize(
            vmin=0.0,
            vmax=max_val
        )

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data and legend_labels:
        patches = [
            mpatches.Patch(
                color=legend_colors[index],
                label=legend_labels[index]
            )
            for index in range(len(legend_labels))
        ]
        legend = ax.legend(
            handles=patches,
            title="Absolute Difference",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    elif not is_binary_data:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.7
        )
        colorbar.set_label(
            "Absolute Difference",
            fontsize=12,
            rotation=270,
            labelpad=20
        )
        colorbar.set_ticks(
            np.linspace(0.0, max_val, num=6)
        )

    ax.set_title(
        f"Change Absolute Difference - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"change_absolute_difference_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path
def plot_change_difference_sign_v10_map(raster_path):
    # Plot and export the Change Difference (Eₙ, Eq. 53 v10) map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    if masked_map.count() > 0:
        min_obs = int(np.floor(np.ma.min(masked_map)))
        max_obs = int(np.ceil(np.ma.max(masked_map)))
    else:
        min_obs, max_obs = 0, 0

    lo = min(min_obs, -2)
    hi = max(max_obs, 2)
    if lo > 0:
        lo = 0
    if hi < 0:
        hi = 0

    boundaries = np.arange(lo - 0.5, hi + 1.5, 1.0)
    n_classes = len(boundaries) - 1

    base = blue_gray_red_cmap()
    colors = [
        mcolors.to_hex(color)
        for color in base(
            np.linspace(0.0, 1.0, n_classes, endpoint=True)
        )
    ]

    zero_index = -lo if lo <= 0 <= hi else None
    if zero_index is not None and 0 <= zero_index < len(colors):
        colors[zero_index] = "#f2f2f2"

    cmap = ListedColormap(colors)
    cmap.set_bad(color="white")
    norm = BoundaryNorm(boundaries, cmap.N)

    ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    class_values_desc = list(range(hi, lo - 1, -1))
    patches = []
    for value in class_values_desc:
        index = value - lo
        color = colors[index]
        patches.append(
            mpatches.Patch(
                color=color,
                label=str(value)
            )
        )

    legend = ax.legend(
        handles=patches,
        title="Change Difference (Eₙ)",
        loc="center left",
        bbox_to_anchor=(1.05, 0.5),
        frameon=False,
        fontsize=12,
        alignment="left"
    )
    legend.get_title().set_fontsize("14")
    legend.get_title().set_ha("left")
    for item in legend.get_texts():
        item.set_ha("left")

    ax.set_title(
        f"Change Difference (Eₙ, Eq. 53 v10) - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"change_difference_sign_v10_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


def plot_change_absolute_difference_sign_v10_map(raster_path):
    # Plot and export the Change Absolute Difference (Fₙ, Eq. 54 v10) map.
    masked_map, bounds, src_crs, transformer = read_resampled_map(raster_path)
    fig, ax = plt.subplots(
        figsize=(14, 12)
    )

    legend_colors = []
    legend_labels = []

    if is_binary_data:
        unique_vals = (
            sorted(np.unique(masked_map.compressed()).astype(int))
            if masked_map.count() > 0 else [0]
        )
        blue_cmap = plt.get_cmap("Blues")

        if 0 in unique_vals:
            legend_colors.append("#f2f2f2")
            legend_labels.append("0")

        positive_vals = [value for value in unique_vals if value > 0]
        count_pos = len(positive_vals)

        if count_pos > 0:
            blue_shades = [
                blue_cmap(fraction)
                for fraction in np.linspace(0.35, 1.0, count_pos)
            ]
            legend_colors.extend(
                [mcolors.to_hex(color) for color in blue_shades]
            )
            legend_labels.extend([str(value) for value in positive_vals])

        cmap = ListedColormap(
            legend_colors if legend_colors else ["#f2f2f2"]
        )
        cmap.set_bad(color="white")

        if unique_vals:
            boundaries = [value - 0.5 for value in unique_vals] + [unique_vals[-1] + 0.5]
        else:
            boundaries = [-0.5, 0.5]
        norm = BoundaryNorm(boundaries, cmap.N)
    else:
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        cmap = mcolors.LinearSegmentedColormap.from_list(
            "gray_to_blue_abs",
            [
                "#f2f2f2",
                plt.get_cmap("Blues")(1.0)
            ]
        )
        cmap.set_bad(color="white")
        norm = mcolors.Normalize(
            vmin=0.0,
            vmax=max_val
        )

    im = ax.imshow(
        masked_map,
        cmap=cmap,
        norm=norm,
        extent=[
            bounds.left,
            bounds.right,
            bounds.bottom,
            bounds.top
        ]
    )

    prepare_map_axes(
        ax,
        bounds,
        src_crs,
        transformer
    )

    if is_binary_data and legend_labels:
        patches = [
            mpatches.Patch(
                color=legend_colors[index],
                label=legend_labels[index]
            )
            for index in range(len(legend_labels))
        ]
        legend = ax.legend(
            handles=patches,
            title="Absolute Difference (Fₙ)",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            frameon=False,
            fontsize=12,
            alignment="left"
        )
        legend.get_title().set_fontsize("14")
        legend.get_title().set_ha("left")
        for item in legend.get_texts():
            item.set_ha("left")
    elif not is_binary_data:
        colorbar = fig.colorbar(
            im,
            ax=ax,
            orientation="vertical",
            fraction=0.046,
            pad=0.08,
            shrink=0.7
        )
        colorbar.set_label(
            "Absolute Difference (Fₙ)",
            fontsize=12,
            rotation=270,
            labelpad=20
        )
        max_val = float(np.ma.max(masked_map)) if masked_map.count() > 0 else 1.0
        ticks = np.linspace(0.0, max_val, num=min(6, int(max_val) + 1))
        colorbar.set_ticks(ticks)
        colorbar.set_ticklabels([str(int(value)) for value in ticks])

    ax.set_title(
        f"Change Absolute Difference (Fₙ, Eq. 54 v10) - {class_name.capitalize()}",
        fontsize=18,
        pad=20
    )

    png_path = output_path / f"change_absolute_difference_sign_v10_{class_name}_map.png"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    return png_path


map_pngs = {}

map_pngs["presence_hit"] = plot_presence_hit_map(
    raster_outputs["presence_hit"]
)
map_pngs["presence_time_difference"] = plot_presence_time_difference_map(
    raster_outputs["presence_time_difference"]
)
map_pngs["presence_difference"] = plot_presence_difference_map(
    raster_outputs["presence_difference"]
)
map_pngs["presence_absolute_difference"] = plot_presence_absolute_difference_map(
    raster_outputs["presence_absolute_difference"]
)
map_pngs["change_hit"] = plot_change_hit_map(
    raster_outputs["change_hit"]
)
map_pngs["change_difference"] = plot_change_difference_map(
    raster_outputs["change_difference"]
)
map_pngs["change_absolute_difference"] = plot_change_absolute_difference_map(
    raster_outputs["change_absolute_difference"]
)
map_pngs["change_difference_sign_v10"] = plot_change_difference_sign_v10_map(
    raster_outputs["change_difference_sign_v10"]
)
map_pngs["change_absolute_difference_sign_v10"] = plot_change_absolute_difference_sign_v10_map(
    raster_outputs["change_absolute_difference_sign_v10"]
)

print("\nMap exports completed.")
for name, path in map_pngs.items():
    print(f" - {name}: {path}")